# FlashInspector AI – Test Model on Video

Upload a video and run your trained YOLOv8 model on it.

Supports both **detection** (`best_detect.pt` — S1,2,5,6) and **segmentation** (`best_segment.pt` — S3,4) models.

**Steps:** Check GPU → Install deps → Get code → Upload model → Upload video → Run inference → View & download results

## 1. Check GPU

Go to **Runtime → Change runtime type → T4 GPU** if no GPU is detected.

In [ ]:
!nvidia-smi 2>/dev/null || echo "\n\u26a0\ufe0f No GPU detected. Go to Runtime \u2192 Change runtime type \u2192 T4 GPU."

## 2. Install dependencies & clone repo

In [ ]:
!pip install -q ultralytics opencv-python
!git clone --depth 1 https://github.com/patrisiyarum/fire.git /content/fire 2>/dev/null || echo "Repo already cloned."
%cd /content/fire/flashinspector-ai

from pathlib import Path
print("\nProject ready at:", Path.cwd())
print("Model exists:", Path("best.pt").exists())

## 3. Upload your model

Upload the model you trained on Kaggle:
- **`best_detect.pt`** — detection model (S1, S2, S5, S6 + Roboflow: fire blankets, extinguishers, smoke detectors, marking signs, emergency exits, etc.)
- **`best_segment.pt`** — segmentation model (S3, S4: fire extinguisher condition/occlusion)

If the repo already has `best.pt`, it will be used as a fallback.

In [ ]:
from pathlib import Path
from ultralytics import YOLO
from google.colab import files

# Upload your model from Kaggle (best_detect.pt or best_segment.pt)
print("Upload your model file (best_detect.pt or best_segment.pt):")
print("(Click 'Cancel' to use best.pt from the repo if available)\n")

try:
    up = files.upload()
    MODEL_PATH = Path(list(up.keys())[0])
except Exception:
    # Fallback to repo model
    MODEL_PATH = Path("best.pt")
    if not MODEL_PATH.exists():
        raise FileNotFoundError("No model found. Upload best_detect.pt or best_segment.pt from Kaggle.")

model = YOLO(str(MODEL_PATH))
task = model.overrides.get("task", "detect")
print(f"\nModel loaded: {MODEL_PATH}")
print(f"Task: {task}")
print(f"Classes: {model.names}")

## 4. Upload your video

In [ ]:
from google.colab import files
from pathlib import Path

print("Select a video file (.mp4, .avi, .mov, .mkv):")
uploaded = files.upload()
VIDEO_PATH = Path(list(uploaded.keys())[0])
print(f"\nUploaded: {VIDEO_PATH} ({VIDEO_PATH.stat().st_size / 1024 / 1024:.1f} MB)")

## 5. Run inference on video

Adjust the settings below if needed:
- **CONFIDENCE**: minimum detection confidence (0.0–1.0)
- **FRAME_SKIP**: process every Nth frame (higher = faster, lower = more detailed)

In [ ]:
import cv2
import time
from pathlib import Path
from IPython.display import clear_output

# Settings
CONFIDENCE = 0.25
FRAME_SKIP = 5  # process every 5th frame

# Open video
cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"Video: {VIDEO_PATH.name}")
print(f"Resolution: {width}x{height}, FPS: {fps:.1f}, Total frames: {total_frames}")
print(f"Duration: {total_frames / fps:.1f}s")
print(f"Processing every {FRAME_SKIP} frame(s)...\n")

# Output video
out_dir = Path("inference_results")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / f"result_{VIDEO_PATH.stem}.mp4"
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out_writer = cv2.VideoWriter(str(out_path), fourcc, fps / FRAME_SKIP, (width, height))

# Process frames
all_detections = []
frame_idx = 0
processed = 0
start_time = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frame_idx % FRAME_SKIP == 0:
        results = model(frame, conf=CONFIDENCE, verbose=False)[0]

        # Collect detections
        for box in results.boxes:
            all_detections.append({
                "frame": frame_idx,
                "timestamp_sec": round(frame_idx / fps, 2),
                "class": results.names[int(box.cls)],
                "confidence": round(float(box.conf), 3),
            })

        annotated = results.plot()
        out_writer.write(annotated)
        processed += 1

        if processed % 50 == 0:
            elapsed = time.time() - start_time
            pct = frame_idx / total_frames * 100
            print(f"  {pct:.0f}% done — {processed} frames processed ({processed / elapsed:.1f} fps)")

    frame_idx += 1

cap.release()
out_writer.release()

elapsed = time.time() - start_time
print(f"\nDone! Processed {processed} frames in {elapsed:.1f}s ({processed / max(elapsed, 0.01):.1f} fps)")
print(f"Annotated video saved: {out_path}")

## 6. Detection summary

In [ ]:
if not all_detections:
    print("No detections found in the video.")
else:
    # Summary by class
    from collections import Counter
    counts = Counter(d["class"] for d in all_detections)
    print(f"Model type: {task}")
    print(f"Total detections: {len(all_detections)}\n")
    print("Detections by class:")
    for cls, count in counts.most_common():
        confs = [d["confidence"] for d in all_detections if d["class"] == cls]
        print(f"  {cls}: {count} detections (avg confidence: {sum(confs)/len(confs):.2f})")

    if task == "segment":
        seg_frames = len(set(d["frame"] for d in all_detections))
        print(f"\nSegmentation masks were drawn on {seg_frames} frames.")

    # Show first few detections with timestamps
    print(f"\nFirst 10 detections:")
    for d in all_detections[:10]:
        print(f"  [{d['timestamp_sec']}s] {d['class']}: {d['confidence']}")

## 7. Preview a sample frame

In [ ]:
import cv2
from IPython.display import display, Image as IPImage
from pathlib import Path
import tempfile

# Open the annotated video and grab a frame from the middle
cap = cv2.VideoCapture(str(out_path))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)  # jump to middle
ret, frame = cap.read()
cap.release()

if ret:
    tmp = Path(tempfile.mktemp(suffix=".jpg"))
    cv2.imwrite(str(tmp), frame)
    display(IPImage(str(tmp), width=800))
    print(f"Showing frame {total // 2} of {total}")
else:
    print("Could not read a frame from the output video.")

## 8. Download the annotated video

In [ ]:
from google.colab import files
from pathlib import Path

if Path(out_path).exists():
    print(f"Downloading: {out_path} ({Path(out_path).stat().st_size / 1024 / 1024:.1f} MB)")
    files.download(str(out_path))
else:
    print("Output video not found. Run the inference cell above first.")